In [ ]:
import oxDNA_analysis_tools
import oxpy

In [ ]:
from oxDNA_analysis_tools.UTILS.RyeReader import describe, get_confs

In [ ]:
#from oxDNA_analysis_tools.UTILS import oxview
import oxpy
import multiprocessing

In [ ]:
!python3 -m pip show oxpy

In [ ]:
import oxpy
import multiprocessing

# Running oxpy multiple times from the same jupyter kernel crashes the kernel for some reason
# So we will be starting simulations from a separate thread.
def spawn(f, kwargs = {}):
    p = multiprocessing.Process(target=f, kwargs = kwargs)
    p.start()
    return p

# Sets up and runs the simulation
def run(**kwargs):
    with oxpy.Context():
        inp = oxpy.InputFile()
        inp.init_from_filename("input")

        # Modify inputfile at runtime
        # all input parameters provided to oxDNA need to be strings 
        for k,v in kwargs.items(): 
            inp[k] = v
        
        manager = oxpy.OxpyManager(inp)
        manager.run_complete()

p = None

In [ ]:
inp = oxpy.InputFile()

In [ ]:
#top_info, traj_info =describe("../Tile_DimerRelaxed_ForcesRemoved.top","../trajectory_DimertStructure.dat")
top_info, traj_info =describe("../Monomer_ForcesRemoved.top","../trajectory_MonomerStructure.dat")
#top_info1, traj_info1 =describe("../Monomer_ForcesRemoved.top","../trajectory_MonomerStructure.dat")

In [ ]:
with open("../Monomer_ForcesRemoved.top") as f:
    n_bases = int(f.readline().strip())

print("Number of bases:", n_bases)

In [ ]:
from oxDNA_analysis_tools.mean import mean
from oxDNA_analysis_tools.deviations import deviations
from oxDNA_analysis_tools.UTILS.RyeReader import describe

# Get the trajectory and topolo

In [ ]:
from oxDNA_analysis_tools.mean import mean
from oxDNA_analysis_tools.deviations import deviations
from oxDNA_analysis_tools.UTILS.RyeReader import describe

# Get the trajectory and topology file names from the oxpy.InputFile object
#top = inp["../Tile_DimerRelaxed_ForcesRemoved.top"]
#traj = inp["../trajectory_DimertStructure.dat"]
#top_info, traj_info = describe(top, traj)

# Compute the mean structure and RMSFs
mean_conf = mean(traj_info, top_info, ncpus=4)
RMSDs, RMSFs = deviations(traj_info, top_info, mean_conf, ncpus=4)

#They come out as numpy arrays, need to be a dict with a list for visualization
RMSFs = {"RMSF": RMSFs.tolist()}

In [ ]:
14231

In [ ]:
top_info, traj_info =describe("../Monomer_ForcesRemoved.top","../Aligned_Monomer.dat")


In [ ]:
from oxDNA_analysis_tools.UTILS.RyeReader import linear_read

configs=linear_read(traj_info, top_info)

In [ ]:
times = list()
coords = list()
for frame in linear_read(traj_info,top_info):
    # frame is a dict-like object, containing:
    # frame["time"]
    # frame["N"]
    # frame["positions"]
    # frame["velocities"] (if available)
    # frame["orientation"] (if available)
    for f in range(len(frame)):
        pos = frame[f].positions   # shape (N, 3)
        coords.append(pos)
        times.append(frame[f].time)

coords = np.stack(coords)     # (n_frames, N, 3)
times  = np.array(times)

print("coords shape:", coords.shape)
print("times shape:", times.shape)

In [ ]:
coords_sel=coords[:,[14038,14183,13814],:]

In [ ]:
oxDNA_analysis_tools.distance.distance([traj_info],[top_info]

In [ ]:
distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13894]], p2ss = [[13895]], ncpus=1)

In [ ]:
idxs = [14038,
13991,
13990,
13943,
13942,
13895,
13894,
13847,
13088,
13041,
13814,
13769,
14086,
14039,
14134,
14087,
13698,
13651,
14182,
14135,
14230,
14183]

In [ ]:
cmap = plt.get_cmap("tab20")   # 20 distinct colors
centroids = list()
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
#idxs = [14038,13991,13990,13943,13942,13895,13894]
# Plot each index in a different color
for i, idx in enumerate(idxs):
    pts = coords[:, idx, :]          # shape (T, 3)
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    ax.scatter(
        x, y, z,
        s=10,
        color=cmap(i % 20),          # wrap if >20 indices
        label=f"index {idx}",
        alpha=0.7
    )

    centroid = pts.mean(axis=0)      # shape (3,)
    centroids.append(centroid)

    # plot centroid as a larger marker in same color
    ax.scatter(
        centroid[0], centroid[1], centroid[2],
        s=80,
        color=cmap(i % 20),
        edgecolor="k",
        marker="X",
        alpha=1.0
    )

centroids = np.vstack(centroids)     # shape (M, 3), M = len(idxs)

diff = centroids[:, None, :] - centroids[None, :, :]   # (M, M, 3)
dist_matrix = np.linalg.norm(diff, axis=-1)            # (M, M)

# upper-triangular distances (i < j) for summary stats
M = len(idxs)
iu, ju = np.triu_indices(M, k=1)
pairwise_dists = dist_matrix[iu, ju]

avg_dist = pairwise_dists.mean()
#print("Average distance between centroids:", avg_dist)

#print("\nPairwise centroid distances:")
#for i_u, j_u, d in zip(iu, ju, pairwise_dists):
#    print(f"idx {idxs[i_u]} – idx {idxs[j_u]}: {d:.3f}")


ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("3D scatter of selected indices across trajectory")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
cmap = plt.get_cmap("tab20")   # 20 distinct colors
centroids = list()
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
#idxs = [14038,13991,13990,13943,13942,13895,13894]
# Plot each index in a different color
for i, idx in enumerate(idxs):
    pts = coordsShifted_Aligned[:, idx, :]          # shape (T, 3)
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    ax.scatter(
        x, y, z,
        s=10,
        color=cmap(i % 20),          # wrap if >20 indices
        label=f"index {idx}",
        alpha=0.7
    )

    centroid = pts.mean(axis=0)      # shape (3,)
    centroids.append(centroid)

    # plot centroid as a larger marker in same color
    ax.scatter(
        centroid[0], centroid[1], centroid[2],
        s=80,
        color=cmap(i % 20),
        edgecolor="k",
        marker="X",
        alpha=1.0
    )

#cmap = plt.get_cmap("tab20")   # 20 distinct colors
#centroids = list()
#fig = plt.figure(figsize=(9, 7))
#ax = fig.add_subplot(111, projection='3d')
#idxs = [14038,13991,13990,13943,13942,13895,13894]
# Plot each index in a different color
for i, idx in enumerate(idxs):
    pts = coordsShifted1[:, idx, :]          # shape (T, 3)
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    ax.scatter(
        x, y, z,
        s=10,
        color=cmap(i % 20),          # wrap if >20 indices
        label=f"index {idx}",
        alpha=0.7
    )

    centroid = pts.mean(axis=0)      # shape (3,)
    centroids.append(centroid)

    # plot centroid as a larger marker in same color
    ax.scatter(
        centroid[0], centroid[1], centroid[2],
        s=80,
        color=cmap(i % 20),
        edgecolor="k",
        marker="X",
        alpha=1.0
    )

centroids = np.vstack(centroids)     # shape (M, 3), M = len(idxs)

diff = centroids[:, None, :] - centroids[None, :, :]   # (M, M, 3)
dist_matrix = np.linalg.norm(diff, axis=-1)            # (M, M)

# upper-triangular distances (i < j) for summary stats
M = len(idxs)
iu, ju = np.triu_indices(M, k=1)
pairwise_dists = dist_matrix[iu, ju]

avg_dist = pairwise_dists.mean()
#print("Average distance between centroids:", avg_dist)

#print("\nPairwise centroid distances:")
#for i_u, j_u, d in zip(iu, ju, pairwise_dists):
#    print(f"idx {idxs[i_u]} – idx {idxs[j_u]}: {d:.3f}")


ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("3D scatter of selected indices across trajectory")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.0))
plt.tight_layout()
plt.show()
centroids = np.vstack(centroids)     # shape (M, 3), M = len(idxs)

diff = centroids[:, None, :] - centroids[None, :, :]   # (M, M, 3)
dist_matrix = np.linalg.norm(diff, axis=-1)            # (M, M)

# upper-triangular distances (i < j) for summary stats
M = len(idxs)
iu, ju = np.triu_indices(M, k=1)
pairwise_dists = dist_matrix[iu, ju]

avg_dist = pairwise_dists.mean()
#print("Average distance between centroids:", avg_dist)

#print("\nPairwise centroid distances:")
#for i_u, j_u, d in zip(iu, ju, pairwise_dists):
#    print(f"idx {idxs[i_u]} – idx {idxs[j_u]}: {d:.3f}")


ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("3D scatter of selected indices across trajectory")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Choose distinct colors
colors = ["red", "blue", "green", "purple"]

scatter3D(ax, shiftPts1, label="shiftPts1", color=colors[0])
scatter3D(ax, ShiftPoints_Aligned, label="shiftPts2", color=colors[1])
scatter3D(ax, coordsShifted_Aligned[:,idxs[7],:], label="Extension 1", color=colors[2])
scatter3D(ax, coordsShifted1[:,idxs[15],:], label="Extension 2", color=colors[3])

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D Scatter of Four Different Coordinate Sets")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.00))

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

In [ ]:
kde=gaussian_kde(shiftPts1.T)

In [ ]:
xs = np.linspace(x.min(), x.max(), 200)
ys = np.linspace(y.min(), y.max(), 200)
zs = np.linspace(z.min(), z.max(), 200)
X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')

# Flatten for evaluation
grid = np.vstack([X.ravel(), Y.ravel(), Z.ravel()])

PDF = kde(grid).reshape(X.shape)

In [ ]:
# Pick a threshold for "high probability"
threshold = np.percentile(PDF, 97)   # top 3%

mask = PDF > threshold
Xh = X[mask]
Yh = Y[mask]
Zh = Z[mask]

fig = plt.figure(figsize=(8,7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Xh, Yh, Zh, s=5, c='red', alpha=0.5)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('3D High-Density Regions of Effective KDE PDF')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

# pts1, pts2 are (N, 3)
pts1 = ShiftPoints_Aligned
pts2 = shiftPts1

# Build KDE objects
kde1 = gaussian_kde(pts1.T)
kde2 = gaussian_kde(pts2.T)

all_pts = np.vstack([pts1, pts2])
mins = all_pts.min(axis=0)
maxs = all_pts.max(axis=0)

# Grid resolution
g = 100  # increase for smoother, slower

xs = np.linspace(mins[0]-5, maxs[0]+5, g)
ys = np.linspace(mins[1]-5, maxs[1]+5, g)
zs = np.linspace(mins[2]-5, maxs[2]+5, g)
dx = xs[1]-xs[0]
dy = ys[1]-ys[0]
dz = zs[1]-zs[0]
dV = dx*dy*dz
X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')
grid = np.vstack([X.ravel(), Y.ravel(), Z.ravel()])   # shape (3, g^3)

pdf1 = kde1(grid).reshape((g, g, g))
pdf2 = kde2(grid).reshape((g, g, g))

# Effective concentration = product PDF
eff_pdf = pdf1 * pdf2

# Pick a threshold for "high probability"
threshold = np.percentile(eff_pdf, 97)   # top 3%

mask = eff_pdf > threshold
Xh = X[mask]
Yh = Y[mask]
Zh = Z[mask]

fig = plt.figure(figsize=(8,7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Xh, Yh, Zh, s=5, c='red', alpha=0.5)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('3D High-Density Regions of Effective KDE PDF')
plt.tight_layout()
plt.show()

In [ ]:
c_eff_geom = (eff_pdf * dV).sum() 

In [ ]:
c_eff_geom = (eff_pdf * dV).sum() 
c0 = 0.602   # 1 M in nm^-3
#c0 = 1
DeltaG_conf_kBT = -np.log(c_eff_geom / 5e-9)

In [ ]:
c_eff_geom

In [ ]:
DeltaG_conf_kBT

In [ ]:
print("∫pdf1 dV ≈", (pdf1 * dV).sum())
print("∫pdf2 dV ≈", (pdf2 * dV).sum())

In [ ]:
eff_pdf

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Choose distinct colors
colors = ["red", "purple"]

scatter3D(ax, shiftPts1, label="shiftPts1", color=colors[0])
scatter3D(ax, ShiftPoints_Aligned, label="shiftPts2", color=colors[1])
plt.show()

In [ ]:
import numpy as np

def pdf3d_hist(points, edges=None, nbins=40):
    points = np.asarray(points, float)

    if edges is None:
        mins = points.min(axis=0)
        maxs = points.max(axis=0)
        edges = [
            np.linspace(mins[d], maxs[d], nbins + 1)
            for d in range(3)
        ]

    H, edges = np.histogramdd(
        sample=points,
        bins=edges,
        density=True   # <-- constant PDF normalization
    )
    return H, edges

In [ ]:
all_pts = np.vstack([shiftPts1, ShiftPoints_Aligned])
mins = all_pts.min(axis=0)
maxs = all_pts.max(axis=0)

nbins = 40
edges = [
    np.linspace(mins[d], maxs[d], nbins + 1)
    for d in range(3)
]


H1,edges1=pdf3d_hist(shiftPts1)
H2,edges2=pdf3d_hist(ShiftPoints_Aligned)

In [ ]:
dx = edges[0][1] - edges[0][0]
dy = edges[1][1] - edges[1][0]
dz = edges[2][1] - edges[2][0]
dV = dx * dy * dz

print("∫pdf1 dV ≈", H1.sum() * dV)
print("∫pdf2 dV ≈", H2.sum() * dV)

In [ ]:
shiftPts1

In [ ]:
def scatter3D(ax, coords, label, color, size=8, alpha=0.7):
    """
    coords can be (T, N, 3) or (M, 3).
    This flattens everything into a single Nx3 cloud.
    """
    pts = np.reshape(coords, (-1, 3))   # works for any shape (..., 3)
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    ax.scatter(x, y, z, s=size, c=color, alpha=alpha, label=label)

In [ ]:
pairs = [
    (0,21),(1,20),(2,19),(3,18),(4,17),(5,16),(6,15),(7,14),(8,13),
    (9,12),(10,11),(11,10),(12,9),(13,8),(14,7),(15,6),(16,5),(17,4),(18,3),(19,2),(20,1),(21,0)
]

In [ ]:
pairs[0][0]

In [ ]:
confEnergies = list()
for i in range(len(pairs)):
    pair = pairs[i]
    helix1 = pair[0]
    helix2 = pair[1]

    pts1 = coords[:, idxs[helix1], :]          # Identified Coordinates of extension 1
    x1, y1, z1 = pts1[:, 0], pts1[:, 1], pts1[:, 2]
    centroid1 = pts1.mean(axis=0)      # Identified Centroid of extension 1
    shiftPts1=pts1 - centroid1         # Shifted Extension to be origin
    centroid1x = centroid1[0]
    centroid1y = centroid1[1]
    centroid1z = centroid1[2]
    Shiftedcentroid1 = shiftPts1.mean(axis=0)
    
    pts2 = coords[:, idxs[helix2], :]          # Identified Coordinates of extension 1
    x2, y2, z2 = pts2[:, 0], pts2[:, 1], pts2[:, 2]
    #centroid2 = pts2.mean(axis=0)      #  Identified Centroid of extension 2
    shiftPts2=pts2 - centroid1          # Shifted Extension 2 same as extension 1, relative positions maintained
    shiftPts2 = -shiftPts2              # Flip tile 2
    centroid2 = shiftPts2.mean(axis=0)  # Find Centroid of tile 2
    ShiftPoints_Aligned = shiftPts2 - (centroid2-Shiftedcentroid1)

    coordsShifted1=coords - centroid1 

    coordsShifted=coords - centroid1 
    coordsShifted = -coordsShifted
    coordsShifted_Aligned = coordsShifted - (centroid2-Shiftedcentroid1)

    energies = list()
    for a in range(len(pairs)):
        if(a!=i):
            pair = pairs[a]
            helix1 = pair[0]
            helix2 = pair[1]
            
            pts1 = coordsShifted1[:, idxs[helix1], :]
            pts2 = coordsShifted_Aligned[:, idxs[helix2], :]
            
            # Build KDE objects
            kde1 = gaussian_kde(pts1.T)
            kde2 = gaussian_kde(pts2.T)
            
            all_pts = np.vstack([pts1, pts2])
            mins = all_pts.min(axis=0)
            maxs = all_pts.max(axis=0)
            
            # Grid resolution
            g = 50  # increase for smoother, slower
            
            xs = np.linspace(mins[0]-5, maxs[0]+5, g)
            ys = np.linspace(mins[1]-5, maxs[1]+5, g)
            zs = np.linspace(mins[2]-5, maxs[2]+5, g)
            dx = xs[1]-xs[0]
            dy = ys[1]-ys[0]
            dz = zs[1]-zs[0]
            dV = dx*dy*dz
            X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')
            grid = np.vstack([X.ravel(), Y.ravel(), Z.ravel()])   # shape (3, g^3)
            
            pdf1 = kde1(grid).reshape((g, g, g))
            pdf2 = kde2(grid).reshape((g, g, g))
            
            # Effective concentration = product PDF
            eff_pdf = pdf1 * pdf2
            c_eff_geom = (eff_pdf * dV).sum() 
            c0 = 0.602   # 1 M in nm^-3
            #c0 = 1
            DeltaG_conf_kBT = -np.log(c_eff_geom / 5e-9)
            energies.append(DeltaG_conf_kBT)
    confEnergies.append(energies)

In [ ]:
from matplotlib.colors import LogNorm


In [ ]:
arr1=np.asarray(confEnergies)

In [ ]:
pd.DataFrame(arr1 > 0).value_counts()

In [ ]:
np.asarray(confEnergies)

In [ ]:
#sns.heatmap(confEnergies)
mask = arr1 < 0

#sns.heatmap(confEnergies, mask=mask)
sns.heatmap(arr1,mask=mask ,cmap='viridis', norm=LogNorm(vmin=1, vmax=200))

In [ ]:
mask = arr1 > 0

#sns.heatmap(confEnergies, mask=mask)
sns.heatmap(abs(arr1),mask=mask ,cmap='viridis')

In [ ]:
confEnergies

In [ ]:
energies

In [ ]:
confEnergies

In [ ]:
#for i, idx in enumerate(idxs):
pts1 = coords[:, idxs[3], :]          # Identified Coordinates of extension 1
x1, y1, z1 = pts1[:, 0], pts1[:, 1], pts1[:, 2]
centroid1 = pts1.mean(axis=0)      # Identified Centroid of extension 1
shiftPts1=pts1 - centroid1         # Shifted Extension to be origin
centroid1x = centroid1[0]
centroid1y = centroid1[1]
centroid1z = centroid1[2]
Shiftedcentroid1 = shiftPts1.mean(axis=0)

pts2 = coords[:, idxs[19], :]          # Identified Coordinates of extension 1
x2, y2, z2 = pts2[:, 0], pts2[:, 1], pts2[:, 2]
#centroid2 = pts2.mean(axis=0)      #  Identified Centroid of extension 2
shiftPts2=pts2 - centroid1          # Shifted Extension 2 same as extension 1, relative positions maintained
shiftPts2 = -shiftPts2              # Flip tile 2
centroid2 = shiftPts2.mean(axis=0)  # Find Centroid of tile 2
ShiftPoints_Aligned = shiftPts2 - (centroid2-Shiftedcentroid1)            # Align Bonds

In [ ]:
len(coordsShifted_Aligned[:,0,:])

In [ ]:
coordsShifted1=coords - centroid1 

coordsShifted=coords - centroid1 
coordsShifted = -coordsShifted
coordsShifted_Aligned = coordsShifted - (centroid2-Shiftedcentroid1)

In [ ]:
len(coords[0])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(dist_matrix, cmap="viridis")

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Distance (nm)")

# Tick labels = index IDs
ax.set_xticks(np.arange(len(idxs)))
ax.set_yticks(np.arange(len(idxs)))
ax.set_xticklabels(idxs)
ax.set_yticklabels(idxs)

# Rotate x tick labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

ax.set_title("Centroid–Centroid Distance Heatmap")
ax.set_xlabel("Index")
ax.set_ylabel("Index")

plt.tight_layout()
plt.show()

In [ ]:
points = coords_sel.reshape(-1, 3)

In [ ]:
x = points[:, 0]
y = points[:, 1]
z = points[:, 2]

In [ ]:
nbins = 100  # tweak as needed

# Build a 2D histogram in XY
H, xedges, yedges = np.histogram2d(x, y, bins=nbins)

# Plot
plt.figure(figsize=(6, 5))

# extent: [xmin, xmax, ymin, ymax] so axes show real coordinates
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

plt.imshow(
    H.T, 
    origin="lower",
    extent=extent,
    aspect="equal"
)
plt.colorbar(label="Counts")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Spatial density of selected indices (XY)")
plt.tight_layout()
plt.show()

In [ ]:
coords_sel = coords[:, idxs, :]         

# Flatten over time + indices → a big cloud of points
points = coords_sel.reshape(-1, 3)      # (T*M, 3)

x = points[:, 0]
y = points[:, 1]
z = points[:, 2]

In [ ]:
len(frame)

In [ ]:
configs

In [ ]:
!oat align ../trajectory_MonomerStructure.dat ../Aligned_Monomer.dat

In [ ]:
from oxDNA_analysis_tools import distance
import numpy as np
import pandas as pd

In [ ]:
distance.min_image(14191, 14222, box = 10000)

In [ ]:
distance.min_image(14191, 14046, box = 10000)

Helix 11
14191
14222

Helix 10
14142
14175

Helix 9
13659
13691

Helix 8
14127
14094

Helix 7
14079
14046
Helix 6:
13809
13775

Helix 5:
13081
13048

Helix 4:
13854
13889

Helix 3:
13935
13902

Helix 2:
13983
13951

Helix 1:
14031
13998
[14191,14222,14142,14175,13659,13691,14127,14094,14079,14046,13809,13775,13081,13048,13854,13889,13935,13902,13983,13951,14031,13998]

In [ ]:
p2 = [14191,14222,14142,14175,13659,13691,14127,14094,14079,14046,13809,13775,13081,13048,13854,13889,13935,13902,13983,13951,14031,13998]

In [ ]:
labels_list

In [ ]:
labels = ['Helix 0','Helix 0','Helix 1','Helix 1','Helix 2','Helix 2','Helix 3','Helix 3','Helix 4','Helix 4'
          ,'Helix 5','Helix 5','Helix 6','Helix 6','Helix 7','Helix 7','Helix 8','Helix 8','Helix 9','Helix 9'
          ,'Helix 10','Helix 10']

In [ ]:
labels_list = labels[::-1]

In [ ]:
len(labels_list)

In [ ]:
len(p2)

In [ ]:
dists1 = list()
dists2 = list()
dists3 = list()
dists4 = list()
distMean = list()
labs3 = list()
c=0
for i in p2:
    distances=distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13809]], p2ss = [[i]], ncpus=4)
    distances1=distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13775]], p2ss = [[i]], ncpus=4)
    distances3=distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13854]], p2ss = [[i]], ncpus=4)
    distances4=distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13889]], p2ss = [[i]], ncpus=4)
    distResults=np.array(distances)
    distResults1=np.array(distances1)
    distResults3=np.array(distances3)
    distResults4=np.array(distances4)
    for a in range(len(distResults[0][0])):
        dist1=distResults[0][0][a]
        dist2=distResults1[0][0][a]
        dist3=distResults3[0][0][a]
        dist4=distResults4[0][0][a]
        distMean.append(np.mean([dist1,dist2,dist3,dist4]))
        labs3.append(labels_list[c])
    #dists1.append(dist1)
    #dists2.append(dist2)
    #dists3.append(dist3)
    #dists4.append(dist4)
    c=c+1
#distRes=pd.DataFrame({'Point':p2, 'Distance1':dists1, 'Distance2':dists2, 'Distance3':dists3, 'Distance4':dists4})

In [ ]:
len(distances[0][0])

In [ ]:
distRes=pd.DataFrame({'Label':labs3,'Distance':distMean})

In [ ]:
distMean

In [ ]:
distRes['Helix'] = labels_list

In [ ]:
labU=np.unique(labels)

In [ ]:
subRes=distRes[distRes['Helix'] == labU[0]]

In [ ]:
np.mean(subRes[['Distance1','Distance2']])

In [ ]:
subRes[['Distance1','Distance2','Distance3','Distance4']].to_numpy().flatten()

In [ ]:
labU

In [ ]:
distAve = list()
labelOrder = list()
for i in range(len(labU)):
    subRes=distRes[distRes['Helix'] == labU[i]]
    #distAve.append(np.mean(subRes[['Distance1','Distance2','Distance3','Distance4']]))
    distAve.append(subRes[['Distance1','Distance2','Distance3','Distance4']].to_numpy().flatten())
    rep = [str(labU[i])]*8
    labelOrder.append(rep)

In [ ]:
distRes[distRes['Helix'] == 'Helix 7']

In [ ]:
np.array(labelOrder).flatten()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
len(np.array(labelOrder).flatten())

In [ ]:
len(np.array(distAve).flatten())

In [ ]:
sns.set_context('talk')

In [ ]:
DistanceHelix=pd.DataFrame({'Label':np.array(labelOrder).flatten(),'Distance':np.array(distAve).flatten()})

In [ ]:
DistanceHelix =distRes
d1=DistanceHelix[DistanceHelix['Label'] == 'Helix 0']
#d1=d1[d1['Label'] == 'Helix 0']
d1=d1[d1['Label'] == 'Helix 1']
d1=d1[d1['Label'] == 'Helix 2']
d1=d1[d1['Label'] == 'Helix 7']
d1=d1[d1['Label'] == 'Helix 8']
d1=d1[d1['Label'] == 'Helix 9']

lab1=['Helix 6','Helix 7','Helix 8','Helix 9']
d1 = DistanceHelix[DistanceHelix["Label"].isin(lab1)]


In [ ]:
d1

In [ ]:
d1 = d1

In [ ]:
d1.to_csv('../Tile_Tole_Figures/LabelvsDistance.csv')

In [ ]:
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600

In [ ]:
fig, ax = plt.subplots()

sns.barplot(data= d1, x = 'Label', y = 'Distance', order = ['Helix 6','Helix 7','Helix 8','Helix 9'])
plt.xticks(rotation = 45)
ax.set_xticklabels(['0','+1','+2','+3'])
plt.ylabel('Average Distance (nm)')
plt.xlabel('Bond Position')
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/LabelvsDistance.svg')

In [ ]:
distRes

In [ ]:
distances=distance.distance(traj_infos = [traj_info], top_infos= [top_info], p1ss = [[13809,13775]], p2ss = [[13081]], ncpus=4)

In [ ]:
np.array(distances).shape

In [ ]:
distances

In [ ]:
distances[1]

In [ ]:
distResults=np.array(distances)
distResults[0][0][0]

In [ ]:
distResults[0][0][0]

In [ ]:
distResults[0][1][0]

In [ ]:
from oxDNA_analysis_tools.UTILS.oxview import oxdna_conf

# Display python objects in an oxview iframe
oxdna_conf(top_info, mean_conf, RMSFs)

In [ ]:
RMSFs.to_json()

In [ ]:
import json
with open("outputMonomer.json", "w") as f:
    json.dump(RMSFs, f)